In [2]:
"""
PFD‑Net V1 — 二维 Poisson 方程求解（test-B：纯 APINN，无高频子网）
==================================================================
PDE:   -Δu(x,y) = f(x,y),   (x,y) ∈ Ω = [-1,1]²
边界:   u = g_b(x,y),        (x,y) ∈ ∂Ω   (Dirichlet)

精确解: F(x,y) = g(x) + g(y),  g(x) = exp(-x²) sin(μx²)
源项:   f(x,y) = -(g''(x) + g''(y))
边界值: g_b = F(x,y)|∂Ω

网络:     纯 APINN（低频网络），固定傅里叶频率（非可学习）
对照组:   test-B - 无高频子网，仅 APINN 单层网络
训练:     单阶段 15000 epoch，使用 PINN 损失（PDE残差 + BC）
保存:     权重文件 + 每个记录点的 res_loss, bc_loss, total_loss, l2_error
"""

import os
import time
import warnings
import numpy as np
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# ── 设备 ──────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
torch.manual_seed(42)
np.random.seed(42)

CHECKPOINT_DIR = "checkpoints_test_B_pde"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MU = 30.0
LAMBDA_R = 1.0      # PDE残差权重
LAMBDA_B = 10.0     # 边界条件权重
pi = torch.tensor(np.pi, dtype=torch.float64, device=device)


# ==============================================================================
# 精确解 & 源项（用于生成源项和边界条件）
# ==============================================================================

def g_func(x: torch.Tensor) -> torch.Tensor:
    """g(x) = exp(-x²) sin(μx²)"""
    return torch.exp(-x ** 2) * torch.sin(MU * x ** 2)


def g_second(x: torch.Tensor) -> torch.Tensor:
    """
    g''(x) = e^{-x²} [(4x² - 4μ²x² - 2) sin(μx²) + (2μ - 8μx²) cos(μx²)]
    """
    ex = torch.exp(-x ** 2)
    s = torch.sin(MU * x ** 2)
    c = torch.cos(MU * x ** 2)
    coeff_s = 4 * x**2 - 4 * MU**2 * x**2 - 2
    coeff_c = 2 * MU - 8 * MU * x**2
    return ex * (coeff_s * s + coeff_c * c)


def u_exact(xy: torch.Tensor) -> torch.Tensor:
    """精确解 F(x,y) = g(x) + g(y)；xy: (N,2) → (N,1)"""
    return g_func(xy[:, 0:1]) + g_func(xy[:, 1:2])


def f_source(xy: torch.Tensor) -> torch.Tensor:
    """源项 f，满足 -Δu = f：f = -(g''(x) + g''(y))"""
    return -(g_second(xy[:, 0:1]) + g_second(xy[:, 1:2]))


def g_boundary(xy: torch.Tensor) -> torch.Tensor:
    """边界条件"""
    return u_exact(xy)


# ==============================================================================
# 网络结构（保持不变）
# ==============================================================================

class FixedFourierEmbed2D(nn.Module):
    """固定傅里叶嵌入（频率不参与训练），输出维度 = 4 * n_scales"""
    def __init__(self, scales):
        super().__init__()
        self.register_buffer(
            "scales",
            torch.tensor(scales, dtype=torch.float64)
        )

    def get_scales(self):
        return self.scales.detach().cpu().numpy()

    def forward(self, xy):
        feats = []
        for s in self.scales:
            feats += [
                torch.sin(2 * pi * s * xy[:, 0:1]),
                torch.cos(2 * pi * s * xy[:, 0:1]),
                torch.sin(2 * pi * s * xy[:, 1:2]),
                torch.cos(2 * pi * s * xy[:, 1:2]),
            ]
        return torch.cat(feats, dim=-1)


class APINN2D(nn.Module):
    """低频主网络（固定频率）"""
    def __init__(self, hidden_dim=128, num_layers=4, low_scales=(1, 2, 4)):
        super().__init__()
        self.embed = FixedFourierEmbed2D(low_scales)
        in_dim = 2 + 4 * len(low_scales)
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.layers.append(nn.Linear(hidden_dim, 1))
        for m in self.layers:
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, xy):
        h = torch.cat([xy, self.embed(xy)], dim=-1)
        for layer in self.layers[:-1]:
            h = torch.tanh(layer(h))
        return self.layers[-1](h)

    def print_scales(self, prefix=""):
        print(f"{prefix}APINN 固定频率: {np.round(self.embed.get_scales(), 2)}")


# ==============================================================================
# 自动微分：Δu = u_xx + u_yy
# ==============================================================================

def laplacian(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    """
    xy: (N,2)，调用前已设 requires_grad=True
    返回: Δu = u_xx + u_yy，shape (N,1)
    """
    u = model(xy)

    grad_u = torch.autograd.grad(
        u, xy,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]                                    # (N,2)

    u_xx = torch.autograd.grad(
        grad_u[:, 0:1], xy,
        grad_outputs=torch.ones_like(grad_u[:, 0:1]),
        create_graph=True,
        retain_graph=True,
    )[0][:, 0:1]                            # (N,1)

    u_yy = torch.autograd.grad(
        grad_u[:, 1:2], xy,
        grad_outputs=torch.ones_like(grad_u[:, 1:2]),
        create_graph=True,
        retain_graph=True,
    )[0][:, 1:2]                            # (N,1)

    return u_xx + u_yy


# ==============================================================================
# 采样（改为PINN需要的内部点和边界点）
# ==============================================================================

def sample_interior(n: int, seed: int = 1) -> torch.Tensor:
    """
    [-1,1]² 内部混合采样（均匀网格 + 随机），更好地覆盖高频振荡区域。
    返回普通张量（不设 requires_grad，由训练循环 clone 后设置）。
    """
    rng = np.random.RandomState(seed)
    side = int(np.sqrt(n // 2))
    gx = np.linspace(-1 + 1e-4, 1 - 1e-4, side)
    gy = np.linspace(-1 + 1e-4, 1 - 1e-4, side)
    GX, GY = np.meshgrid(gx, gy)
    xy_grid = np.stack([GX.ravel(), GY.ravel()], axis=1)
    n_rand = n - xy_grid.shape[0]
    xy_rand = rng.uniform(-1.0, 1.0, size=(n_rand, 2))
    pts = np.concatenate([xy_grid, xy_rand], axis=0)
    return torch.tensor(pts, dtype=torch.float64, device=device)


def sample_boundary(n_per_edge: int, seed: int = 2) -> torch.Tensor:
    """四条边各 n_per_edge 个点，返回 (4*n_per_edge, 2)"""
    rng = np.random.RandomState(seed)
    t = rng.uniform(-1.0, 1.0, n_per_edge)
    bottom = np.stack([t, -np.ones(n_per_edge)], axis=1)
    top = np.stack([t, np.ones(n_per_edge)], axis=1)
    left = np.stack([-np.ones(n_per_edge), t], axis=1)
    right = np.stack([np.ones(n_per_edge), t], axis=1)
    pts = np.concatenate([bottom, top, left, right], axis=0)
    return torch.tensor(pts, dtype=torch.float64, device=device)


# ==============================================================================
# PINN 损失计算
# ==============================================================================

def pinn_loss(model: nn.Module,
              xy_int: torch.Tensor,
              f_int: torch.Tensor,
              xy_bc: torch.Tensor,
              u_bc: torch.Tensor,
              lambda_r: float,
              lambda_b: float):
    """
    计算 PDE残差损失 + 边界损失，返回 (total, loss_r, loss_b)
    """
    # PDE 残差：-Δu - f = 0
    xy_r = xy_int.clone().requires_grad_(True)
    lap_u = laplacian(model, xy_r)              # Δu
    loss_r = torch.mean((-lap_u - f_int) ** 2)  # -Δu - f = 0

    # 边界条件
    loss_b = nn.functional.mse_loss(model(xy_bc), u_bc)

    total = lambda_r * loss_r + lambda_b * loss_b
    return total, loss_r, loss_b


# ==============================================================================
# 单阶段训练（test-B PDE版本：纯 APINN，使用PINN损失）
# ==============================================================================

def train_test_B_pde(
    # 网络超参
    hidden_dim=128,
    num_layers=4,
    low_scales=(1, 2, 4),
    # PINN配点
    n_interior=10000,
    n_per_edge=250,
    # 训练轮次
    total_epochs=15000,
    # 学习率
    lr=5e-3,
    # 损失权重
    lambda_r=LAMBDA_R,
    lambda_b=LAMBDA_B,
    # 日志
    log_every=1,
):
    print(f"\n{'='*60}")
    print(f"  test-B PDE版本：纯APINN求解Poisson方程 | {total_epochs} epochs")
    print(f"  PDE: -Δu = f,  Ω=[-1,1]²")
    print(f"  λ_r={lambda_r}, λ_b={lambda_b}")
    print(f"{'='*60}")

    # ── 固定配点（内部点 + 边界点）──────────────────────────────────────────
    xy_int = sample_interior(n_interior, seed=1)   # 内部点
    f_int = f_source(xy_int).detach()              # 源项，固定

    xy_bc = sample_boundary(n_per_edge, seed=2)    # 边界点
    u_bc = g_boundary(xy_bc).detach()              # 边界真值

    print(f"内部配点: {xy_int.shape[0]}")
    print(f"边界配点: {xy_bc.shape[0]}")

    # ── 测试集（100×100 均匀网格，与精确解对比）────────────────────────────
    nx = 100
    xv, yv = np.meshgrid(np.linspace(-1, 1, nx), np.linspace(-1, 1, nx))
    xy_test = torch.tensor(
        np.stack([xv.ravel(), yv.ravel()], axis=1),
        dtype=torch.float64, device=device
    )
    u_test = u_exact(xy_test).detach()
    u_test_sq_mean = torch.mean(u_test ** 2).item()

    # ── 辅助：评估测试集 ─────────────────────────────────────────────────────
    def eval_test(model):
        with torch.no_grad():
            pred = model(xy_test)
            l2 = torch.sqrt(
                torch.mean((pred - u_test) ** 2) / u_test_sq_mean
            ).item()
        return l2

    # ── 全局记录 ─────────────────────────────────────────────────────────────
    epochs_record, res_losses, bc_losses, total_losses, l2_errors = [], [], [], [], []

    t0 = time.time()

    # ── 构建 APINN ───────────────────────────────────────────────────────────
    apinn = APINN2D(hidden_dim, num_layers, low_scales).double().to(device)
    total_p = sum(p.numel() for p in apinn.parameters())
    print(f"APINN 参数量: {total_p:,}")
    print(f"固定频率: {list(low_scales)}  学习率 lr = {lr:.2e}")

    opt = torch.optim.Adam(apinn.parameters(), lr=lr)
    # sch = torch.optim.lr_scheduler.CosineAnnealingLR(
    #     opt, total_epochs, eta_min=1e-5)

    # ── 训练循环（使用PINN损失）──────────────────────────────────────────────
    for ep in range(1, total_epochs + 1):
        apinn.train()
        opt.zero_grad()

        # 计算PINN损失
        loss, loss_r, loss_b = pinn_loss(
            apinn, xy_int, f_int, xy_bc, u_bc, lambda_r, lambda_b
        )
        loss.backward()
        nn.utils.clip_grad_norm_(apinn.parameters(), 1.0)
        opt.step()
        # sch.step()

        # 记录
        if ep % log_every == 0 or ep == 1:
            l2 = eval_test(apinn)
            epochs_record.append(ep)
            res_losses.append(loss_r.item())
            bc_losses.append(loss_b.item())
            total_losses.append(loss.item())
            l2_errors.append(l2)

            if ep % max(1, log_every * 5) == 0 or ep == 1:
                print(f"  epoch {ep:6d} | res={loss_r.item():.3e} "
                      f"bc={loss_b.item():.3e} total={loss.item():.3e} "
                      f"L2={l2:.6f} | {time.time()-t0:.1f}s")

    # ── 最终评估 ──────────────────────────────────────────────────────────────
    with torch.no_grad():
        final_pred = apinn(xy_test)
        final_l2 = torch.sqrt(
            torch.mean((final_pred - u_test) ** 2) / u_test_sq_mean
        ).item()
        final_mae = torch.mean(torch.abs(final_pred - u_test)).item()
    total_time = time.time() - t0

    print(f"\n训练完成")
    print(f"  test-B PDE | L2={final_l2:.6f} | MAE={final_mae:.6f} "
          f"| 时间={total_time:.1f}s")
    apinn.print_scales("  ")

    # ── 保存 ─────────────────────────────────────────────────────────────────
    torch.save(apinn.state_dict(),
               os.path.join(CHECKPOINT_DIR, "testB_pde_weights.pt"))
    np.savez(
        os.path.join(CHECKPOINT_DIR, "testB_pde_curves.npz"),
        epochs=np.array(epochs_record),
        res_loss=np.array(res_losses),
        bc_loss=np.array(bc_losses),
        total_loss=np.array(total_losses),
        l2_error=np.array(l2_errors),
    )
    print(f"权重与曲线已保存至 {CHECKPOINT_DIR}/")

    return dict(
        model=apinn,
        xy_test=xy_test, u_test=u_test,
        epochs=np.array(epochs_record),
        res_loss=np.array(res_losses),
        bc_loss=np.array(bc_losses),
        total_loss=np.array(total_losses),
        l2_error=np.array(l2_errors),
        fl2=final_l2, fmae=final_mae,
    )


# ==============================================================================
if __name__ == "__main__":
    train_test_B_pde(
        hidden_dim=128,
        num_layers=4,
        low_scales=(1, 2, 4),
        n_interior=10000,
        n_per_edge=250,
        total_epochs=15000,
        lr=5e-3,
        lambda_r=1.0,
        lambda_b=10.0,
        log_every=1,
    )

使用设备: cuda

  test-B PDE版本：纯APINN求解Poisson方程 | 15000 epochs
  PDE: -Δu = f,  Ω=[-1,1]²
  λ_r=1.0, λ_b=10.0
内部配点: 10000
边界配点: 1000
APINN 参数量: 35,073
固定频率: [1, 2, 4]  学习率 lr = 5.00e-03
  epoch      1 | res=7.383e+05 bc=4.490e-01 total=7.383e+05 L2=1.270470 | 0.0s
  epoch      5 | res=6.561e+05 bc=2.959e-01 total=6.561e+05 L2=1.011722 | 0.2s
  epoch     10 | res=4.649e+05 bc=4.189e-01 total=4.650e+05 L2=1.031755 | 0.5s
  epoch     15 | res=3.359e+05 bc=4.725e-01 total=3.359e+05 L2=0.984986 | 0.7s
  epoch     20 | res=2.793e+05 bc=3.997e-01 total=2.793e+05 L2=1.044053 | 0.9s
  epoch     25 | res=1.829e+05 bc=6.603e-01 total=1.829e+05 L2=1.086099 | 1.2s
  epoch     30 | res=8.270e+04 bc=8.425e-01 total=8.271e+04 L2=1.293568 | 1.4s
  epoch     35 | res=5.104e+04 bc=1.003e+00 total=5.105e+04 L2=1.259349 | 1.6s
  epoch     40 | res=2.868e+04 bc=1.043e+00 total=2.869e+04 L2=1.141138 | 1.9s
  epoch     45 | res=1.707e+04 bc=9.076e-01 total=1.707e+04 L2=1.064568 | 2.1s
  epoch     50 | res=1.073e